# TabM: Fixed-Configuration Results

**Purpose.** Explain TabM and summarize its completed fixed-configuration experiments with 6, 10, 19, and 25 factors. Compare the added information from each factor group using the same sample and validation design.

This is an executed report of saved experiments. Running the notebook verifies and summarizes those experiments without training new models.

**What is TabM?**

TabM is a neural network for tables, where each row is an observation and each column is a feature. Its building block is a multilayer perceptron (MLP): layers that combine inputs and apply nonlinear transformations. TabM produces several predictions for the same row through a set of jointly trained members that share most of their weights. This provides an ensemble within one model, with fewer separate parameters than training the same number of independent networks. The design is described by [Gorishniy, Kotelnikov, and Babenko, ICLR 2025](https://arxiv.org/abs/2410.24210).

In these experiments, each model has **32 members**. Each member produces a satisfaction probability, and their arithmetic mean is the final prediction. The 32 members belong to one training run; the three random seeds used below are separate repetitions. The [official implementation](https://github.com/yandex-research/tabm) supports numerical features and categorical features, including internal one-hot encoding.

**Why is it a suitable candidate for this dataset?**

The integrated data combine numerical measures such as age and salary with categorical survey responses and work-value ratings. TabM can use both after preprocessing. Its nonlinear layers can represent relationships that depend on several inputs together, for example whether the association between salary and satisfaction varies with how much someone values salary. This is a modeling possibility, not an interaction established by this report.

The shared ensemble makes repeated fits across factor groups and validation folds practical; all four experiments were completed locally on CPU. These are reasons to evaluate TabM here, not evidence that it is the best possible model. Whether the additional factor groups help is assessed from held-out predictions below.

## 0. Setup

Locate the project, load the four saved protocols, and check that they describe the same fixed configuration. The original runs used scikit-learn 1.4.2 and TabM 0.0.3; reproducing the split-generation assertions below requires the original scikit-learn behavior. This reporting notebook does not require PyTorch or TabM.

Saved outputs are included for reading. Re-execution requires the local experiment artifacts and the NSCG codebook.

In [1]:
from pathlib import Path
from itertools import combinations
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import sklearn
from IPython.display import HTML, Markdown, display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
)

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'data/nscg').is_dir() and (p / 'personal/outputs').is_dir()
)
stage_dirs = {
    6: ROOT / 'personal/outputs/tabm6_integration63_20261111',
    10: ROOT / 'personal/outputs/tabm10_stageB_integration63_20261111',
    19: ROOT / 'personal/outputs/tabm19_stageC_integration63_20261111',
    25: ROOT / 'personal/outputs/tabm25_stageD_integration63_20261111',
}
protocols = {n: json.loads((p / 'protocol.json').read_text()) for n, p in stage_dirs.items()}
config = protocols[6]['config']
SEEDS = config['seeds']
PRIMARY_SEED = config['primary_seed']
DATA = ROOT / protocols[6]['data_path']
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()

for n, protocol in protocols.items():
    assert len(protocol['features']) == n
    assert protocol['config'] == config
    assert protocol['config']['hyperparameter_search'] is False
    assert protocol['data_sha256'] == protocols[6]['data_sha256']
    assert protocol['fold_sha256'] == protocols[6]['fold_sha256']
    assert protocol['target'] == protocols[6]['target']
    assert protocol['threshold_satisfied'] == 0.5
    assert protocol['survey_weighted'] is False
assert sha256(DATA) == protocols[6]['data_sha256']

def preview(title, frame):
    # Keep raw previews available without interrupting the main report.
    display(HTML(
        '<details><summary>' + title + '</summary><div style="overflow-x:auto">'
        + frame.head().to_html(index=False) + '</div></details>'
    ))

def table(frame, caption, formats=None):
    display(
        frame.style.hide(axis='index').format(formats or {})
        .set_caption(caption)
        .set_table_styles([
            {'selector': 'caption', 'props': [('text-align', 'left'), ('font-weight', 'bold'), ('margin-bottom', '10px')]},
            {'selector': 'th', 'props': [('background-color', '#edf3f7'), ('text-align', 'left'), ('padding', '8px')]},
            {'selector': 'td', 'props': [('padding', '8px'), ('border-bottom', '1px solid #e5e7eb')]},
        ])
    )

print('Four fixed protocols verified. Hyperparameter search: False.')
print(f'Reporting environment: scikit-learn {sklearn.__version__}')

Four fixed protocols verified. Hyperparameter search: False.
Reporting environment: scikit-learn 1.4.2


**Observation.** The four protocols specify identical model and training hyperparameters. Factor groups change the input information and therefore the input dimension; they do not change the prespecified hidden-layer configuration.

## 1. Train, validation, and test sets

The target is binary job satisfaction: `JOBSATIS` codes 1 or 2 become **satisfied = 1**; codes 3 or 4 become **unsatisfied = 0**, following the saved experiment protocol. `OBSNUM` identifies a respondent, and `N3OCPR` identifies the occupation group. Neither identifier is a model input.

In [2]:
data = pd.read_csv(DATA, low_memory=False)
preview('Raw integrated data: first five rows', data)
assert data.shape == (protocols[6]['rows'], protocols[6]['columns'])
assert data.OBSNUM.is_unique
assert data.JOBSATIS.isin([1, 2, 3, 4]).all()
assert data[protocols[25]['features']].notna().all().all()
y = data.JOBSATIS.isin([1, 2]).astype(int).to_numpy()
groups = data.N3OCPR.to_numpy()
print(f'Sample: {len(data):,} respondents in {data.N3OCPR.nunique()} occupation groups.')
print(f'Satisfied: {y.sum():,} ({y.mean():.2%}); unsatisfied: {(1-y).sum():,} ({(1-y).mean():.2%}).')
print('Missing values across the 25 included factors: 0.')

OBSNUM,REFYR,AGE,SEX_2023,LFSTAT,JOBSATIS,N3OCPR,N3OCPRX,N2OCPRMG,N3OCPRNG,OCEDRLP,DGRDG,WTSURVY,SUPWK,WAPRI,FACADV,FACBEN,FACCHAL,FACIND,FACLOC,FACRESP,FACSAL,FACSEC,FACSOC,NSCG_OCC_TITLE,SOC2018_CODE,ONET_SOC_CODE,ONET_OCC_TITLE,ONET_PROFILE_ID,MAPPING_BASIS,MAPPING_TYPE,MAPPING_SCOPE_NOTE,MAPPING_COVERAGE,MAPPING_IS_PROXY,MAPPING_PROXY_TYPE,ONET_CANDIDATE_COUNT,ONET_RATED_COUNT,ONET_RATING_COVERAGE,INTEGRATION_STATUS,ONET_WV_VERSION,ONET_WV_SCALE,ONET_WV_DATE_MIN,ONET_WV_DATE_MAX,ONET_WV_SOURCE,ONET_WV_achievement,ONET_WV_working_conditions,ONET_WV_recognition,ONET_WV_relationships,ONET_WV_support,ONET_WV_independence,SALARY,HRSWK,EMSECSM,EMSIZE,SATADV,SATBEN,SATCHAL,SATIND,SATLOC,SATRESP,SATSAL,SATSEC,SATSOC
2023000002,2023,45,F,1,2,782230,782230,7,78,3,1,221.4500,N,7,1,2,1,1,2,1,2,2,1,"Other service occupations, except health (e.g., personal care service occupations, building and grounds cleaning and maintenance occupations, etc.)",NaN,NaN,"Group mean: Other service occupations, except health (e.g., personal care service occupations, building and grounds cleaning and maintenance occupations, etc.)",NSCG_782230,Reviewed study occupation mapping,group_mean_partial,SOC37 building/grounds maintenance plus SOC39 personal care; exclude healthcare support.,partial,False,none,43,36,0.837209,matched_reviewed,30.2,EX,2008-06,2020-11,Analyst; Analyst - Transition,3.051667,3.053611,2.532500,5.018333,3.795556,3.856111,12000,28,3,1,1,4,1,1,1,1,3,3,1
2023000003,2023,32,M,1,2,621440,611140,6,62,1,2,110.5786,Y,8,1,1,2,1,1,2,1,2,1,Medical and health services managers,11-9111,11-9111.00,Medical and Health Services Managers,NSCG_621440,Reviewed study occupation mapping,single_SOC,Use the broader management profile; NSCG specifies managers of other managers.,full,False,none,1,1,1.000000,matched_reviewed,30.2,EX,2008-06,2008-06,Analyst,5.000000,6.000000,4.670000,6.000000,5.670000,6.000000,64798,45,2,3,2,2,2,2,2,2,3,1,2
2023000004,2023,59,M,1,1,632530,632530,6,63,1,2,169.9971,N,13,1,1,1,1,1,1,1,1,1,"Teachers: Secondary - computer, math, or sciences",25-2031,25-2031.00,"Secondary School Teachers, Except Special and Career/Technical Education",NSCG_632530,Reviewed study occupation mapping,single_SOC_broader_profile,Use general secondary-school teaching; possible vocational teaching is a mapping assumption.,full,True,broader_profile,1,1,1.000000,matched_reviewed,30.2,EX,2008-06,2008-06,Analyst,5.670000,5.330000,4.000000,7.000000,4.670000,5.000000,141000,40,1,4,2,1,2,1,1,1,2,1,1
2023000005,2023,30,F,1,2,640530,640530,6,64,2,1,42.4254,N,6,1,1,1,1,1,1,1,1,1,"Computer programmers (business, scientific, process control)",15-1251,15-1251.00,Computer Programmers,NSCG_640530,Reviewed study occupation mapping,single_SOC,Listed programming fields use Computer Programmers.,full,False,none,1,1,1.000000,matched_reviewed,30.2,EX,2008-06,2008-06,Analyst,5.670000,5.500000,4.330000,2.330000,5.000000,5.000000,125000,40,3,6,2,2,2,2,1,2,2,2,2
2023000007,2023,25,M,1,2,721510,711470,7,72,1,2,97.7008,Y,1,1,2,1,1,2,1,1,1,3,"Financial specialists (e.g., accountants, auditors)",NaN,NaN,"Group mean: Financial specialists (e.g., accountants, auditors)",NSCG_721510,Reviewed study occupation mapping,group_mean_partial,All 15 detailed SOC13-2000 occupations; 13-2099 is only the residual member.,partial,False,none,15,11,0.733333,matched_reviewed,30.2,EX,2008-06,2020-11,Analyst; Analyst - Transition,4.423636,4.279091,4.121818,4.682727,4.289091,4.771818,82800,40,3,6,1,2,1,1,2,1,3,1,2


Sample: 74,016 respondents in 123 occupation groups.
Satisfied: 66,655 (90.05%); unsatisfied: 7,361 (9.95%).
Missing values across the 25 included factors: 0.


**Observation.** All four experiments use the same 74,016 respondents. Unsatisfied respondents are the minority class, so the results include average precision, precision, recall, and F1 for that class.

**The split is by occupation, not by individual row.** For each outer fold, the occupations in the training, validation, and test sets are disjoint. This evaluates transfer to held-out occupation groups. It also keeps respondents who share the same occupation-level O*NET scores together.

1. **Outer test set:** divide the data into five occupation-grouped folds using `StratifiedGroupKFold(5, shuffle=True, random_state=42)`. Hold one fold out for evaluation and use the other four as the outer training pool.
2. **Inner training and validation:** within that pool, take the first split from `StratifiedGroupKFold(5, shuffle=True, random_state=2026 + outer_fold)`. Fit the network on inner training rows and use validation AUC to select the training duration. There is one inner validation holdout per outer fold, not a full five-fold inner search.
3. **Refit:** reinitialize the network, refit preprocessing on the entire outer training pool, and train on that pool for the selected number of epochs. Validation rows now contribute to the final fit; the outer test rows remain excluded.
4. **Test:** predict the held-out outer fold once. Repeat for all five outer folds. Each respondent receives one out-of-fold (OOF) prediction per seed and factor combination.

Stratification attempts to preserve class proportions while keeping occupation groups intact. Unequal group sizes mean the row counts are not an exact 64% / 16% / 20% division. See the [scikit-learn 1.4 documentation](https://scikit-learn.org/1.4/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html).

In [3]:
memberships = {}
for n, folder in stage_dirs.items():
    memberships[n] = pd.read_csv(folder / 'split_membership.csv')
    preview(f'{n} factors: saved split membership', memberships[n])
    pd.testing.assert_frame_equal(memberships[n], memberships[6])
membership = memberships[6]
assert not membership.duplicated(['outer_fold', 'OBSNUM']).any()

# Recreate the original splits and compare IDs with the saved membership.
outer = StratifiedGroupKFold(5, shuffle=True, random_state=42)
split_rows = []
for fold, (pool_idx, test_idx) in enumerate(outer.split(data, y, groups.astype(str))):
    inner = StratifiedGroupKFold(5, shuffle=True, random_state=2026 + fold)
    a, b = next(inner.split(data.iloc[pool_idx], y[pool_idx], groups[pool_idx]))
    indices = {
        'inner_train': pool_idx[a],
        'inner_validation': pool_idx[b],
        'outer_test': test_idx,
    }
    current = membership.loc[membership.outer_fold.eq(fold)]
    assert len(current) == len(data)
    for role, idx in indices.items():
        saved_ids = current.loc[current.role.eq(role), 'OBSNUM'].to_numpy()
        np.testing.assert_array_equal(saved_ids, data.OBSNUM.iloc[idx].to_numpy())
    for left, right in combinations(indices, 2):
        assert set(groups[indices[left]]).isdisjoint(groups[indices[right]])
    split_rows.append({
        'Outer fold': fold + 1,
        'Train n': len(indices['inner_train']),
        'Validation n': len(indices['inner_validation']),
        'Test n': len(test_idx),
        'Train occupations': len(set(groups[indices['inner_train']])),
        'Validation occupations': len(set(groups[indices['inner_validation']])),
        'Test occupations': len(set(groups[test_idx])),
    })
test_membership = membership.loc[membership.role.eq('outer_test')]
assert test_membership.OBSNUM.is_unique and len(test_membership) == len(data)
splits = pd.DataFrame(split_rows)
table(splits, 'Actual split sizes, identical for all four factor combinations',
      {c: '{:,}' for c in splits.columns if c != 'Outer fold'})

OBSNUM,N3OCPR,outer_fold,role
2023000002,782230,0,inner_train
2023000003,621440,0,inner_train
2023000004,632530,0,inner_train
2023000005,640530,0,inner_train
2023000007,721510,0,inner_train


OBSNUM,N3OCPR,outer_fold,role
2023000002,782230,0,inner_train
2023000003,621440,0,inner_train
2023000004,632530,0,inner_train
2023000005,640530,0,inner_train
2023000007,721510,0,inner_train


OBSNUM,N3OCPR,outer_fold,role
2023000002,782230,0,inner_train
2023000003,621440,0,inner_train
2023000004,632530,0,inner_train
2023000005,640530,0,inner_train
2023000007,721510,0,inner_train


OBSNUM,N3OCPR,outer_fold,role
2023000002,782230,0,inner_train
2023000003,621440,0,inner_train
2023000004,632530,0,inner_train
2023000005,640530,0,inner_train
2023000007,721510,0,inner_train


Outer fold,Train n,Validation n,Test n,Train occupations,Validation occupations,Test occupations
1,"47,024","14,318","12,674",80,18,25
2,"45,788","9,155","19,073",80,19,24
3,"53,088","11,689","9,239",78,21,24
4,"51,191","9,734","13,091",79,20,24
5,"47,061","7,016","19,939",78,19,26


**Observation.** In outer fold 1, 47,024 respondents are used for inner training, 14,318 for validation, and 12,674 for testing. After selecting an epoch count, the refit uses all 61,342 outer-training respondents. These counts are derived from the saved membership and reproduced by the split code above.

Preprocessing is fitted on the inner training set during early stopping, and fitted afresh on the full outer training pool during refitting. Test rows are never used to fit preprocessing or choose epochs. The outer folds have nevertheless been examined across multiple development experiments; they are not a separate, never-viewed final test set.

## 2. Factor combinations

A factor means an original input column, before one-hot encoding. The four completed experiments add groups cumulatively. No individual-factor contribution or causal effect is isolated by this design.

In [4]:
stage_labels = {
    6: ('A', 'Original factors', 'Personal background and current work'),
    10: ('B', 'Work conditions', 'Annual salary, weekly hours, employer sector and size'),
    19: ('C', 'Personal work values', 'Importance attached to nine job characteristics'),
    25: ('D', 'Occupation work values', 'Six occupation-level O*NET work-value scores'),
}
numeric_factors = set(protocols[25]['numeric_features'])
categorical_factors = set(protocols[25]['categorical_features'])
factor_rows = []
previous, included_groups, included_labels = [], [], []
for n, protocol in protocols.items():
    features = protocol['features']
    assert features[:len(previous)] == previous
    stage, label, meaning = stage_labels[n]
    included_groups.append(stage)
    included_labels.append(label)
    numerical_count = sum(f in numeric_factors for f in features)
    categorical_count = sum(f in categorical_factors for f in features)
    assert numerical_count + categorical_count == n
    if 'numeric_features' in protocol:
        assert set(features) & numeric_factors == set(protocol['numeric_features'])
        assert set(features) & categorical_factors == set(protocol['categorical_features'])
    factor_rows.append({
        'Total factors': n,
        'Composition': ' + '.join(included_groups),
        'Groups included': ' + '.join(included_labels),
        'Numerical inputs': numerical_count,
        'Categorical inputs': categorical_count,
    })
    previous = features
table(pd.DataFrame(factor_rows), 'Full composition and model input types for each experiment')

Total factors,Composition,Groups included,Numerical inputs,Categorical inputs
6,A,Original factors,1,5
10,A + B,Original factors + Work conditions,3,7
19,A + B + C,Original factors + Work conditions + Personal work values,3,16
25,A + B + C + D,Original factors + Work conditions + Personal work values + Occupation work values,9,16


**Observation.** The combinations are cumulative: **6 = A; 10 = A + B; 19 = A + B + C; 25 = A + B + C + D**. The tables below list every factor in groups A through D, its meaning, its measurement type, and its actual model preprocessing.

A numerical storage code does not make a survey response a numerical model input. Binary and nominal responses are categories; relatedness, employer-size bands, and importance ratings have ordered response levels but were also encoded as categories in these experiments.

Labels and response-scale identifiers are read from the local NSCG data dictionary. The ordered versus nominal distinctions follow the response definitions in the accompanying codebook. O*NET labels are read from the published Work Values file. Sources: [NSCG 2023 public-use documentation](https://ncses.nsf.gov/surveys/national-survey-college-graduates/2023) and [O*NET 30.2 Work Values](https://www.onetcenter.org/dictionary/30.2/text/work_values.html).

In [5]:
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='Workbook contains no default style')
    dictionary = pd.read_excel(
        ROOT / 'data/nscg/pcg23Public/Dpcg23.xlsx',
        usecols=['SAS_NAME', 'DESCRIPTION', 'SAS_FMT'],
    )
preview('NSCG data dictionary: first five rows', dictionary)
onet = pd.read_csv(
    ROOT / 'data/onet/db_30_2_text/Work Values.txt', sep='\t',
    usecols=['Element Name', 'Scale ID'],
)
preview('O*NET work values dictionary: first five rows', onet)
dictionary_by_name = dictionary.drop_duplicates('SAS_NAME').set_index('SAS_NAME')
onet_names = onet.loc[onet['Scale ID'].eq('EX'), 'Element Name'].drop_duplicates()
onet_labels = {'ONET_WV_' + name.lower().replace(' ', '_'): name for name in onet_names}

# Response-scale interpretations checked against Ppcg23.html.
ordered_formats = {
    '$RELATED_CODES': 'Ordinal category (relatedness)',
    '$EMPLOYER_SIZE_R1_': 'Ordinal category (size band)',
    '$IMPORTANT_CODES': 'Ordinal category (importance)',
}
binary_formats = {'$SEX_CODES', '$YES_NO_CODES'}
previous = []
for n, protocol in protocols.items():
    stage, group_label, _ = stage_labels[n]
    added = protocol['features'][len(previous):]
    rows = []
    for feature in added:
        is_onet = feature.startswith('ONET_WV_')
        label = onet_labels[feature] if is_onet else dictionary_by_name.loc[feature, 'DESCRIPTION']
        if feature in numeric_factors:
            source_type = 'Numerical score' if is_onet else 'Numerical measurement'
            preprocessing = 'log1p, then standardize' if feature == 'SALARY' else 'Standardize'
        else:
            response_format = dictionary_by_name.loc[feature, 'SAS_FMT']
            if response_format in ordered_formats:
                source_type = ordered_formats[response_format]
            elif response_format in binary_formats:
                source_type = 'Binary category'
            else:
                source_type = 'Nominal category'
            source_type += f' ({data[feature].nunique()} observed levels)'
            preprocessing = 'One-hot encode'
        rows.append({
            'Factor': feature, 'Meaning': label,
            'Measurement type': source_type, 'TabM preprocessing': preprocessing,
        })
    used_in = ', '.join(str(size) for size, other in protocols.items() if set(added).issubset(other['features']))
    display(Markdown(f'**Group {stage}: {group_label} ({len(added)} factors).** Included in the {used_in}-factor experiments.'))
    table(pd.DataFrame(rows), f'Group {stage}: complete factor list')
    previous = protocol['features']

SAS_NAME,DESCRIPTION,SAS_FMT
ACADV,Reason for taking courses during reference week: increase advancement opportunities,$YES_NO_CODES
ACCAR,Reason for taking courses during reference week: further education before career,$YES_NO_CODES
ACCCEP,School related costs for course taking paid by employer,$YES_NO_CODES
ACCHG,Reason for taking courses during reference week: change academic/occupational field,$YES_NO_CODES
ACDRG,Type of degree working on during reference week,$DEGREE_CODES


Element Name,Scale ID
Achievement,EX
Working Conditions,EX
Recognition,EX
Relationships,EX
Support,EX


**Group A: Original factors (6 factors).** Included in the 6, 10, 19, 25-factor experiments.

Factor,Meaning,Measurement type,TabM preprocessing
AGE,Age,Numerical measurement,Standardize
SEX_2023,Sex at birth,Binary category (2 observed levels),One-hot encode
DGRDG,Highest degree type,Nominal category (4 observed levels),One-hot encode
OCEDRLP,Extent that principal job is related to highest degree,Ordinal category (relatedness) (3 observed levels),One-hot encode
SUPWK,Supervised others in principal job indicator,Binary category (2 observed levels),One-hot encode
WAPRI,Work activity spent most hours on in principal job,Nominal category (14 observed levels),One-hot encode


**Group B: Work conditions (4 factors).** Included in the 10, 19, 25-factor experiments.

Factor,Meaning,Measurement type,TabM preprocessing
SALARY,Salary (annualized),Numerical measurement,"log1p, then standardize"
HRSWK,Principal job: hours per week typically worked,Numerical measurement,Standardize
EMSECSM,Employer sector (summary codes),Nominal category (3 observed levels),One-hot encode
EMSIZE,Employer size,Ordinal category (size band) (8 observed levels),One-hot encode


**Group C: Personal work values (9 factors).** Included in the 19, 25-factor experiments.

Factor,Meaning,Measurement type,TabM preprocessing
FACADV,Importance of job's opportunities for advancement,Ordinal category (importance) (4 observed levels),One-hot encode
FACBEN,Importance of job's benefits,Ordinal category (importance) (4 observed levels),One-hot encode
FACCHAL,Importance of job's intellectual challenge,Ordinal category (importance) (4 observed levels),One-hot encode
FACIND,Importance of job's degree of independence,Ordinal category (importance) (4 observed levels),One-hot encode
FACLOC,Importance of job's location,Ordinal category (importance) (4 observed levels),One-hot encode
FACRESP,Importance of job's level of responsibility,Ordinal category (importance) (4 observed levels),One-hot encode
FACSAL,Importance of job's salary,Ordinal category (importance) (4 observed levels),One-hot encode
FACSEC,Importance of job's security,Ordinal category (importance) (4 observed levels),One-hot encode
FACSOC,Importance of job's contribution to society,Ordinal category (importance) (4 observed levels),One-hot encode


**Group D: Occupation work values (6 factors).** Included in the 25-factor experiments.

Factor,Meaning,Measurement type,TabM preprocessing
ONET_WV_achievement,Achievement,Numerical score,Standardize
ONET_WV_working_conditions,Working Conditions,Numerical score,Standardize
ONET_WV_recognition,Recognition,Numerical score,Standardize
ONET_WV_relationships,Relationships,Numerical score,Standardize
ONET_WV_support,Support,Numerical score,Standardize
ONET_WV_independence,Independence,Numerical score,Standardize


**Observation.** The four combinations contain **1 numerical + 5 categorical**, **3 numerical + 7 categorical**, **3 numerical + 16 categorical**, and **9 numerical + 16 categorical** inputs, respectively. These counts refer to original factors, not the expanded number of one-hot columns.

The nine `FAC*` fields describe what the person considers important. Their responses are ordered importance categories, but the model uses one-hot encoding and does not assume equal numerical spacing between levels. The six `ONET_WV_*` fields are occupation-level numerical scores and are standardized. The personal and occupation variables are separate sets of dimensions, not a direct one-to-one match; no explicit person-job fit score was constructed.

Standardization and categorical encoding are fitted only on the relevant training subset, as described in Section 1. TabM's categorical indices select one-hot vectors, so ordinal response codes are not treated as continuous values.

The inputs exclude respondent IDs, occupation IDs, survey weights, mapping metadata, and the nine `SAT*` component-satisfaction responses. The 16-factor contrast and 34-factor auxiliary experiment are outside the scope of this report.

## 3. Fixed configuration: no hyperparameter search

**These results are untuned in the sense that network depth, width, learning rate, dropout, and other training settings were fixed before the four runs.** They are a compact experimental configuration, not an assertion that every package default was used.

Validation still selects the epoch count separately for each outer fold, factor set, and seed. Training stops after ten epochs without a validation-AUC improvement greater than 0.0001, or at 100 epochs. The chosen duration is then used for the outer-training refit. Thus, “no hyperparameter tuning” does not mean “no validation-based selection.”

In [6]:
settings = [
    ('Architecture', config['arch_type']),
    ('Hidden blocks / width', f"{config['n_blocks']} / {config['d_block']}"),
    ('Ensemble members', config['k']),
    ('Activation / dropout', f"{config['activation']} / {config['dropout']}"),
    ('Numerical embeddings', 'None' if config['num_embeddings'] is None else config['num_embeddings']),
    ('Optimizer / learning rate', f"{config['optimizer']} / {config['learning_rate']}"),
    ('Weight decay / batch size', f"{config['weight_decay']} / {config['batch_size']}"),
    ('Maximum epochs / patience', f"{config['max_epochs']} / {config['patience']}"),
    ('Minimum validation-AUC improvement', config['min_delta']),
    ('Primary seed / additional seeds', f"{PRIMARY_SEED} / {', '.join(map(str, SEEDS[1:]))}"),
    ('Hyperparameter search', config['hyperparameter_search']),
]
table(pd.DataFrame(settings, columns=['Setting', 'Fixed value']), 'Shared TabM configuration')

training = {}
for n, folder in stage_dirs.items():
    training[n] = pd.read_csv(folder / 'training_summary.csv')
    preview(f'{n} factors: original training summary', training[n])
    assert len(training[n]) == 5 * len(SEEDS)
    assert not training[n].duplicated(['seed', 'fold']).any()
    assert set(map(tuple, training[n][['seed', 'fold']].to_numpy())) == {
        (seed, fold) for seed in SEEDS for fold in range(5)
    }
    assert training[n].selected_epochs.between(1, config['max_epochs']).all()
    check = training[n].loc[training[n].seed.eq(PRIMARY_SEED)].sort_values('fold')
    for saved_col, split_col in [('inner_train_n', 'Train n'), ('inner_val_n', 'Validation n'), ('outer_test_n', 'Test n')]:
        np.testing.assert_array_equal(check[saved_col], splits[split_col])
epoch_table = pd.DataFrame({
    f'{n} factors': frame.loc[frame.seed.eq(PRIMARY_SEED)].sort_values('fold').selected_epochs.to_numpy()
    for n, frame in training.items()
})
epoch_table.insert(0, 'Outer fold', np.arange(1, 6))
table(epoch_table, f'Validation-selected epochs for the primary seed ({PRIMARY_SEED})')

Setting,Fixed value
Architecture,tabm
Hidden blocks / width,2 / 128
Ensemble members,32
Activation / dropout,ReLU / 0.1
Numerical embeddings,None
Optimizer / learning rate,AdamW / 0.002
Weight decay / batch size,0.0003 / 1024
Maximum epochs / patience,100 / 10
Minimum validation-AUC improvement,0.000100
Primary seed / additional seeds,"42 / 43, 44"


seed,fold,selected_epochs,epochs_checked,best_inner_auc,stopped_by_patience,outer_train_n,outer_test_n,inner_train_n,inner_val_n,parameter_count,seconds,roc_auc,ap_unsatisfied,precision_unsatisfied,recall_unsatisfied,f1_unsatisfied,brier_satisfied
42,0,6,16,0.647226,True,61342,12674,47024,14318,45152,154.233690,0.632001,0.148952,0.131499,0.564726,0.213325,0.233871
42,1,5,15,0.625667,True,54943,19073,45788,9155,45152,133.267846,0.628761,0.168902,0.154418,0.554854,0.241598,0.240538
42,2,8,18,0.641772,True,64777,9239,53088,11689,45152,199.278576,0.622403,0.174153,0.155643,0.480691,0.235148,0.223780
42,3,2,12,0.642830,True,60925,13091,51191,9734,45152,109.343565,0.638845,0.175611,0.152678,0.542670,0.238309,0.227781
42,4,3,13,0.659810,True,54077,19939,47061,7016,45152,107.102704,0.627527,0.154753,0.136310,0.454039,0.209673,0.223706


seed,fold,selected_epochs,epochs_checked,best_inner_auc,stopped_by_patience,outer_train_n,outer_test_n,inner_train_n,inner_val_n,parameter_count,seconds,roc_auc,ap_unsatisfied,precision_unsatisfied,recall_unsatisfied,f1_unsatisfied,brier_satisfied
42,0,8,18,0.667852,True,61342,12674,47024,14318,47232,184.036023,0.668321,0.178778,0.145558,0.602085,0.234438,0.229737
42,1,30,40,0.656654,True,54943,19073,45788,9155,47232,491.426119,0.659294,0.191325,0.165114,0.584951,0.257534,0.228519
42,2,10,20,0.681178,True,64777,9239,53088,11689,47232,230.740637,0.656442,0.199609,0.171311,0.506098,0.255975,0.207610
42,3,9,19,0.671524,True,60925,13091,51191,9734,47232,219.879231,0.672176,0.203958,0.160577,0.641138,0.256830,0.228794
42,4,19,29,0.699258,True,54077,19939,47061,7016,47232,321.354816,0.653685,0.172653,0.150343,0.476323,0.228549,0.196278


seed,fold,selected_epochs,epochs_checked,best_inner_auc,stopped_by_patience,outer_train_n,outer_test_n,inner_train_n,inner_val_n,parameter_count,seconds,roc_auc,ap_unsatisfied,precision_unsatisfied,recall_unsatisfied,f1_unsatisfied,brier_satisfied
42,0,12,22,0.703808,True,61342,12674,47024,14318,52992,269.101030,0.707911,0.206575,0.152099,0.692441,0.249413,0.229358
42,1,27,37,0.701674,True,54943,19073,45788,9155,52992,477.738973,0.695466,0.223260,0.169195,0.703883,0.272813,0.243896
42,2,28,38,0.708491,True,64777,9239,53088,11689,52992,546.295527,0.696113,0.227641,0.170174,0.657520,0.270372,0.226744
42,3,9,19,0.703035,True,60925,13091,51191,9734,52992,213.378873,0.701782,0.219580,0.175938,0.666667,0.278404,0.221203
42,4,29,39,0.719793,True,54077,19939,47061,7016,52992,488.923608,0.694087,0.197052,0.167832,0.534819,0.255489,0.190233


seed,fold,selected_epochs,epochs_checked,best_inner_auc,stopped_by_patience,outer_train_n,outer_test_n,inner_train_n,inner_val_n,parameter_count,seconds,roc_auc,ap_unsatisfied,precision_unsatisfied,recall_unsatisfied,f1_unsatisfied,brier_satisfied
42,0,24,34,0.703964,True,61342,12674,47024,14318,53952,347.084715,0.709155,0.209624,0.161938,0.644657,0.258852,0.211363
42,1,27,37,0.700042,True,54943,19073,45788,9155,53952,343.386701,0.694079,0.223194,0.170766,0.700971,0.274629,0.241154
42,2,14,24,0.707809,True,64777,9239,53088,11689,53952,236.033157,0.690285,0.224475,0.175887,0.594512,0.271462,0.213761
42,3,15,25,0.704372,True,60925,13091,51191,9734,53952,234.606308,0.709017,0.228632,0.172842,0.680525,0.275668,0.224921
42,4,49,59,0.720478,True,54077,19939,47061,7016,53952,565.017872,0.693511,0.195818,0.163543,0.568802,0.254043,0.196777


Outer fold,6 factors,10 factors,19 factors,25 factors
1,6,8,12,24
2,5,30,27,27
3,8,10,28,14
4,2,9,9,15
5,3,19,29,49


Training uses binary cross-entropy with class weights computed only from the current training subset: `n / (2 * n_class)`. The loss is averaged across rows and members. Predictions are the mean of the 32 member probabilities. Survey weights are not used in training or evaluation. Class weighting does not itself calibrate the probabilities, and no additional calibration was applied.

## 4. Results table and interpretation

The final table reports **pooled out-of-fold metrics for the prespecified seed 42**. Each score uses predictions from all five outer test folds, not the mean of five fold-level scores. Seeds 43 and 44 are stability checks; the best seed is not selected, and probabilities are not averaged across seeds.

- **ROC AUC:** how well the scores rank satisfied versus unsatisfied respondents.
- **AP (U):** average precision for the unsatisfied class, using `1 - P(satisfied)` as its score. The unsatisfied share printed above is its prevalence reference.
- **Precision (U), recall (U), and F1 (U):** unsatisfied-class metrics at the fixed rule `P(satisfied) >= 0.5` for predicting satisfaction. Precision measures how many predicted-unsatisfied cases are truly unsatisfied; recall measures how many unsatisfied cases are detected; F1 combines the two.
- **AUC 95% interval:** the saved interval from 1,000 occupation-level bootstrap resamples of the fixed OOF predictions. It does not include uncertainty from retraining models.

The following code recomputes the point metrics from saved TabM predictions, checks them against the saved metrics for all three seeds, and reads the existing bootstrap intervals. It does not use any tuning outputs.

In [7]:
metric_cols = ['roc_auc', 'ap_unsatisfied', 'precision_unsatisfied', 'recall_unsatisfied', 'f1_unsatisfied']
prediction_cols = ['OBSNUM', 'N3OCPR', 'fold', 'y_satisfied'] + [f'tabm_seed{s}' for s in SEEDS]
results, seed_auc_rows = [], []
saved_test_fold = test_membership.set_index('OBSNUM').outer_fold.reindex(data.OBSNUM).to_numpy()

for n, folder in stage_dirs.items():
    predictions = pd.read_csv(folder / 'oof_predictions.csv', usecols=prediction_cols)
    preview(f'{n} factors: saved TabM predictions', predictions)
    saved = pd.read_csv(folder / 'model_comparison.csv', usecols=['model', *metric_cols, 'auc_ci_low', 'auc_ci_high'])
    saved = saved.loc[saved.model.str.startswith('tabm_seed')].copy()
    preview(f'{n} factors: saved TabM metrics', saved)
    saved = saved.set_index('model')
    np.testing.assert_array_equal(predictions.OBSNUM, data.OBSNUM)
    np.testing.assert_array_equal(predictions.N3OCPR, groups)
    np.testing.assert_array_equal(predictions.y_satisfied, y)
    np.testing.assert_array_equal(predictions.fold, saved_test_fold)
    assert len(predictions) == len(data) and predictions.OBSNUM.is_unique

    verification = json.loads((folder / 'independent_verification.json').read_text())
    assert verification['status'] == 'passed'
    tabm_checks = [c for c in verification['checks'] if c['model'].startswith('tabm_seed')]
    assert len(tabm_checks) == 15
    assert all(c['max_difference'] <= 1e-7 for c in tabm_checks)

    for seed in SEEDS:
        name = f'tabm_seed{seed}'
        p = predictions[name].to_numpy()
        assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
        predicted_unsatisfied = p < protocols[n]['threshold_satisfied']
        actual_unsatisfied = 1 - y
        measured = {
            'roc_auc': roc_auc_score(y, p),
            'ap_unsatisfied': average_precision_score(actual_unsatisfied, 1 - p),
            'precision_unsatisfied': precision_score(actual_unsatisfied, predicted_unsatisfied, zero_division=0),
            'recall_unsatisfied': recall_score(actual_unsatisfied, predicted_unsatisfied, zero_division=0),
            'f1_unsatisfied': f1_score(actual_unsatisfied, predicted_unsatisfied, zero_division=0),
        }
        np.testing.assert_allclose(
            [measured[c] for c in metric_cols], saved.loc[name, metric_cols].to_numpy(dtype=float),
            rtol=0, atol=1e-12,
        )
        seed_auc_rows.append({'Factors': n, 'Seed': seed, 'ROC AUC': measured['roc_auc']})
        if seed == PRIMARY_SEED:
            results.append({
                'Factors': n,
                'ROC AUC': measured['roc_auc'],
                'AUC 95% interval': f"[{saved.loc[name, 'auc_ci_low']:.4f}, {saved.loc[name, 'auc_ci_high']:.4f}]",
                'AP (U)': measured['ap_unsatisfied'],
                'Precision (U)': measured['precision_unsatisfied'],
                'Recall (U)': measured['recall_unsatisfied'],
                'F1 (U)': measured['f1_unsatisfied'],
            })

results = pd.DataFrame(results)
seed_auc = pd.DataFrame(seed_auc_rows)
paired = pd.read_csv(stage_dirs[25] / 'paired_auc_comparisons.csv')
paired = paired.loc[paired.first_model.str.startswith('tabm_') & paired.reference_model.str.contains('tabm_')].copy()
preview('Saved TabM-only paired AUC comparisons', paired)
delta_25 = paired.set_index('contrast').loc['tabm_seed42_minus_nineteen_factors']
auc = results.set_index('Factors')['ROC AUC']
np.testing.assert_allclose(auc[25] - auc[19], delta_25.auc_difference, rtol=0, atol=1e-12)
print('Verified: four factor sets, identical sample and splits, 60 recorded TabM checkpoint replays, and all 12 OOF metric sets.')

OBSNUM,N3OCPR,fold,y_satisfied,tabm_seed42,tabm_seed43,tabm_seed44
2023000002,782230,4,1,0.375177,0.464998,0.425703
2023000003,621440,4,1,0.598765,0.608316,0.613368
2023000004,632530,3,1,0.617783,0.632005,0.549529
2023000005,640530,4,1,0.450547,0.442061,0.465388
2023000007,721510,4,1,0.579442,0.593028,0.591807


model,roc_auc,ap_unsatisfied,precision_unsatisfied,recall_unsatisfied,f1_unsatisfied,auc_ci_low,auc_ci_high
tabm_seed42,0.630733,0.161865,0.145792,0.519630,0.227699,0.617272,0.643849
tabm_seed43,0.629338,0.159624,0.142955,0.540823,0.226135,0.615502,0.643369
tabm_seed44,0.632165,0.159441,0.143439,0.554680,0.227935,0.618112,0.647102


OBSNUM,N3OCPR,fold,y_satisfied,tabm_seed42,tabm_seed43,tabm_seed44
2023000002,782230,4,1,0.401395,0.406441,0.378538
2023000003,621440,4,1,0.483869,0.452264,0.454924
2023000004,632530,3,1,0.750796,0.726535,0.749823
2023000005,640530,4,1,0.565368,0.529673,0.548483
2023000007,721510,4,1,0.682153,0.657320,0.653452


model,roc_auc,ap_unsatisfied,precision_unsatisfied,recall_unsatisfied,f1_unsatisfied,auc_ci_low,auc_ci_high
tabm_seed42,0.660841,0.184819,0.158074,0.561065,0.246656,0.647430,0.673500
tabm_seed43,0.661946,0.185673,0.152086,0.604809,0.243053,0.648845,0.674701
tabm_seed44,0.661615,0.186324,0.153459,0.591903,0.243728,0.648727,0.674053


OBSNUM,N3OCPR,fold,y_satisfied,tabm_seed42,tabm_seed43,tabm_seed44
2023000002,782230,4,1,0.491513,0.472394,0.431236
2023000003,621440,4,1,0.389876,0.349380,0.367992
2023000004,632530,3,1,0.765876,0.744861,0.712471
2023000005,640530,4,1,0.618834,0.667348,0.597042
2023000007,721510,4,1,0.699529,0.693632,0.645834


model,roc_auc,ap_unsatisfied,precision_unsatisfied,recall_unsatisfied,f1_unsatisfied,auc_ci_low,auc_ci_high
tabm_seed42,0.698646,0.211107,0.167140,0.647738,0.265716,0.687296,0.709339
tabm_seed43,0.699112,0.210897,0.170395,0.627089,0.267975,0.688305,0.709453
tabm_seed44,0.699191,0.212301,0.163655,0.665535,0.262709,0.688155,0.709399


OBSNUM,N3OCPR,fold,y_satisfied,tabm_seed42,tabm_seed43,tabm_seed44
2023000002,782230,4,1,0.550560,0.472621,0.520307
2023000003,621440,4,1,0.370341,0.359246,0.387273
2023000004,632530,3,1,0.715988,0.690927,0.744911
2023000005,640530,4,1,0.571286,0.495249,0.574951
2023000007,721510,4,1,0.697962,0.628160,0.675705


model,roc_auc,ap_unsatisfied,precision_unsatisfied,recall_unsatisfied,f1_unsatisfied,auc_ci_low,auc_ci_high
tabm_seed42,0.699456,0.213813,0.168720,0.641896,0.267206,0.688480,0.709915
tabm_seed43,0.698330,0.212154,0.159917,0.678033,0.258795,0.688068,0.708225
tabm_seed44,0.700270,0.213266,0.166209,0.649776,0.264708,0.690039,0.710193


contrast,first_model,reference_model,auc_difference,ci_low,ci_high
tabm_seed42_minus_six_factors,tabm_seed42,six_tabm_seed42,0.068723,0.061448,0.076642
tabm_seed43_minus_six_factors,tabm_seed43,six_tabm_seed43,0.068992,0.061043,0.078058
tabm_seed44_minus_six_factors,tabm_seed44,six_tabm_seed44,0.068105,0.059954,0.077705
tabm_seed42_minus_ten_factors,tabm_seed42,ten_tabm_seed42,0.038615,0.033732,0.044036
tabm_seed43_minus_ten_factors,tabm_seed43,ten_tabm_seed43,0.036384,0.030975,0.042331


Verified: four factor sets, identical sample and splits, 60 recorded TabM checkpoint replays, and all 12 OOF metric sets.


In [8]:
ranges = seed_auc.groupby('Factors')['ROC AUC'].agg(['min', 'max'])
range_text = '; '.join(f"{n} factors: {row['min']:.4f} to {row['max']:.4f}" for n, row in ranges.iterrows())
display(Markdown(
    f"**Observation.** With the fixed configuration, adding work conditions changes the primary-seed AUC by "
    f"{auc[10]-auc[6]:+.4f}, and adding personal values changes it by {auc[19]-auc[10]:+.4f}. "
    f"Adding occupation values changes AUC by {delta_25.auc_difference:+.4f}; the paired 95% interval is "
    f"[{delta_25.ci_low:+.4f}, {delta_25.ci_high:+.4f}], which includes zero. "
    "This experiment does not establish an additional AUC improvement from 19 to 25 factors.\n\n"
    f"Across the three seeds, the AUC ranges are {range_text}. These ranges describe seed sensitivity, not confidence intervals.\n\n"
    "**Open questions.** Whether tuning changes these comparisons, whether the occupation variables help without the personal-value variables, "
    "and whether the findings hold on a new evaluation sample remain untested here. All results below are development-stage, "
    "fixed-configuration results with validation-based early stopping and no hyperparameter search."
))

**Observation.** With the fixed configuration, adding work conditions changes the primary-seed AUC by +0.0301, and adding personal values changes it by +0.0378. Adding occupation values changes AUC by +0.0008; the paired 95% interval is [-0.0015, +0.0028], which includes zero. This experiment does not establish an additional AUC improvement from 19 to 25 factors.

Across the three seeds, the AUC ranges are 6 factors: 0.6293 to 0.6322; 10 factors: 0.6608 to 0.6619; 19 factors: 0.6986 to 0.6992; 25 factors: 0.6983 to 0.7003. These ranges describe seed sensitivity, not confidence intervals.

**Open questions.** Whether tuning changes these comparisons, whether the occupation variables help without the personal-value variables, and whether the findings hold on a new evaluation sample remain untested here. All results below are development-stage, fixed-configuration results with validation-based early stopping and no hyperparameter search.

In [9]:
table(
    results,
    'TabM only | No hyperparameter tuning | Primary seed 42 | Pooled five-fold OOF results',
    {
        'ROC AUC': '{:.4f}', 'AP (U)': '{:.4f}',
        'Precision (U)': '{:.2%}', 'Recall (U)': '{:.2%}', 'F1 (U)': '{:.4f}',
    },
)

Factors,ROC AUC,AUC 95% interval,AP (U),Precision (U),Recall (U),F1 (U)
6,0.6307,"[0.6173, 0.6438]",0.1619,14.58%,51.96%,0.2277
10,0.6608,"[0.6474, 0.6735]",0.1848,15.81%,56.11%,0.2467
19,0.6986,"[0.6873, 0.7093]",0.2111,16.71%,64.77%,0.2657
25,0.6995,"[0.6885, 0.7099]",0.2138,16.87%,64.19%,0.2672
